In [14]:
import json
from pathlib import Path
import os

import polars as pl
from datasets import Dataset
from huggingface_hub import DatasetCard

In [15]:
OPPORTUNITIES_REPO_ID = "wrmthorne/ukri-funding-opportunities"
MEETINGS_REPO_ID = "wrmthorne/ukri-panel-meetings"

In [16]:
df = pl.read_ndjson(str(Path(".").resolve().parent / "data_cache" / "opportunities" / "extracted_opportunities.jsonl"))

# Drop raw HTML and internal fields not useful for downstream consumers
df = df.drop("raw_html", "metadata_table")

# Convert list columns to comma-separated strings for HF compatibility
df = df.with_columns(
    pl.col("funders").list.join(", "),
    pl.col("co_funders").list.join(", "),
)

df

shape: (2_102, 20)
┌────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ status ┆ funders    ┆ funding_ty ┆ total_fun ┆ … ┆ minimum_a ┆ minimum_f ┆ maximum_f ┆ funding_p │
│ ---    ┆ ---        ┆ pe         ┆ ding      ┆   ┆ ward      ┆ unding_du ┆ unding_du ┆ ercentage │
│ str    ┆ str        ┆ ---        ┆ ---       ┆   ┆ ---       ┆ ration    ┆ ration    ┆ ---       │
│        ┆            ┆ str        ┆ i64       ┆   ┆ i64       ┆ ---       ┆ ---       ┆ i64       │
│        ┆            ┆            ┆           ┆   ┆           ┆ i64       ┆ i64       ┆           │
╞════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ Closed ┆ IUK        ┆ Grant      ┆ 10000000  ┆ … ┆ 10000000  ┆ null      ┆ null      ┆ null      │
│ Closed ┆ IUK        ┆ Grant      ┆ 20000000  ┆ … ┆ null      ┆ null      ┆ null      ┆ null      │
│ Closed ┆ IUK        ┆ Grant      ┆ 1500000   ┆ … ┆ null      ┆ null      ┆ null      ┆ null      │
│ Closed ┆ EPSRC      ┆ Grant      ┆ 2000000   ┆ … ┆ null      ┆ null      ┆ null      ┆ null      │
│ Closed ┆ EPSRC      ┆ Grant      ┆ 4000000   ┆ … ┆ null      ┆ null      ┆ null      ┆ null      │
│ …      ┆ …          ┆ …          ┆ …         ┆ … ┆ …         ┆ …         ┆ …         ┆ …         │
│ Closed ┆ STFC       ┆ Grant      ┆ 2000000   ┆ … ┆ null      ┆ null      ┆ 36        ┆ null      │
│ Closed ┆ ESRC, AHRC ┆ Grant      ┆ 3000000   ┆ … ┆ 350000    ┆ 36        ┆ 36        ┆ 80        │
│ Closed ┆ NERC       ┆ Grant      ┆ 19000000  ┆ … ┆ 2500000   ┆ null      ┆ 48        ┆ 80        │
│ Closed ┆ NERC       ┆ Grant      ┆ 24000000  ┆ … ┆ 3000000   ┆ null      ┆ 48        ┆ 80        │
│ Closed ┆ NERC       ┆ Grant      ┆ 20000000  ┆ … ┆ null      ┆ null      ┆ 48        ┆ 80        │
└────────┴────────────┴────────────┴───────────┴───┴───────────┴───────────┴───────────┴───────────┘

In [4]:
OPPORTUNITIES_DATA_CARD = """\
---
license: cc-by-nc-sa-4.0
language:
- en
tags:
- funding
- ukri
- research-funding
- opportunities
- uk-research
pretty_name: UKRI Funding Opportunities
size_categories:
- 1K<n<10K
---

# UKRI Funding Opportunities

A structured dataset of 2,102 funding opportunities published by [UK Research and Innovation (UKRI)](https://www.ukri.org/opportunity/) across all nine constituent bodies (AHRC, BBSRC, EPSRC, ESRC, IUK, MRC, NERC, RE, STFC, and cross-council UKRI calls).

This dataset accompanies the paper *"Demystifying Funding: Reconstructing a Unified Dataset of the UK Funding Lifecycle"* (NSLP 2026) and the [GtR Database repository](https://github.com/wrmthorne/GtR-Extended).

## Background

UKRI publishes funding opportunities on its website but provides no structured, machine-readable export. Key metadata such as award ranges, total fund values, UKRI contribution percentages, and project durations are frequently stated only within prose, distributed across document sections. This dataset makes that information available in a structured format for research and analysis.

## Collection

All pending, open, and closed opportunities were scraped from the [UKRI Opportunity Finder](https://www.ukri.org/opportunity/) between **12 February 2026 and 22 March 2026** using Playwright browser automation. Raw HTML pages were parsed into a tree-structured representation preserving the document hierarchy (summary, accordion sections, subsections). Because opportunity status and closing dates change over time, this dataset represents a **snapshot** at the time of collection.

## Metadata Extraction

Several fields (`total_funding`, `minimum_award`, `maximum_award`, `funding_percentage`, `minimum_funding_duration`, `maximum_funding_duration`) are not always present in the structured metadata table on the opportunity page. Where absent, they were extracted from the opportunity prose using an LLM-based closed-domain question-answering pipeline. The pipeline uses hierarchical retrieval (hybrid BM25 + dense similarity with branch-deduplication) to select relevant passages, which are then passed to GPT-4o-mini for extraction. Evaluation on 101 manually annotated opportunities (489 question-answer pairs) showed 87.0% overall accuracy. See the accompanying paper for full details.

Fields sourced from the structured metadata table on each opportunity page (`status`, `funders`, `co_funders`, `funding_type`, `publication_date`, `opening_date`, `closing_date`, `href`) are deterministically parsed and are not subject to extraction errors.

## Fields

| Field | Type | Description |
|---|---|---|
| `id` | string | UUID5 identifier derived from the canonical URL |
| `title` | string | Opportunity title |
| `status` | string | Status at time of collection (Closed, Open, Upcoming) |
| `funders` | string | Comma-separated UKRI funder codes (e.g. "EPSRC", "AHRC, ESRC") |
| `co_funders` | string | Comma-separated co-funding organisations, if any |
| `funding_type` | string | Type of funding (e.g. Grant, Fellowship, Studentship) |
| `publication_date` | string | ISO 8601 date the opportunity was published |
| `opening_date` | string | ISO 8601 date the opportunity opened for applications |
| `closing_date` | string | ISO 8601 date the opportunity closed |
| `summary` | string | Plain-text summary of the opportunity |
| `full_text` | string | Full opportunity content as markdown |
| `sections` | list | Hierarchical tree of accordion sections (header, content, children) |
| `updates` | list | Date-text pairs of opportunity updates, if any |
| `href` | string | Canonical URL to the opportunity page |
| `total_funding` | int | Total fund value in GBP (may be LLM-extracted) |
| `minimum_award` | int | Minimum award value in GBP (may be LLM-extracted) |
| `maximum_award` | int | Maximum award value in GBP (may be LLM-extracted) |
| `funding_percentage` | int | UKRI contribution percentage (may be LLM-extracted) |
| `minimum_funding_duration` | int | Minimum project duration in months (may be LLM-extracted) |
| `maximum_funding_duration` | int | Maximum project duration in months (may be LLM-extracted) |

## Limitations

- **Snapshot**: Opportunity statuses and closing dates are as of collection (Feb--Mar 2026) and may have since changed.
- **Coverage**: Not all UKRI-funded activity corresponds to a published opportunity; responsive mode schemes and formula-based allocations are not represented.
- **Extraction accuracy**: LLM-extracted fields have an overall accuracy of 87.0%. The most common errors are field confusion (e.g. returning total funding when asked for maximum award) and hallucination. Maximum award is the hardest field (66.7% accuracy). See the paper for a detailed error analysis.
- **Innovate UK**: IUK opportunities typically publish short summaries linking to the separate Innovation Funding Service, so their `full_text` and `sections` fields contain less detail than other councils.
"""

In [8]:
opp_dataset = Dataset.from_polars(df)
opp_dataset.push_to_hub(OPPORTUNITIES_REPO_ID, token=os.environ["HF_TOKEN_WRITE"])

card = DatasetCard(OPPORTUNITIES_DATA_CARD)
card.push_to_hub(OPPORTUNITIES_REPO_ID, token=os.environ["HF_TOKEN_WRITE"])

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/wrmthorne/ukri-funding-opportunities/commit/c97ee4034aaf6292f38eff0317c90674fefa193e', commit_message='Upload README.md with huggingface_hub', commit_description='', oid='c97ee4034aaf6292f38eff0317c90674fefa193e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wrmthorne/ukri-funding-opportunities', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wrmthorne/ukri-funding-opportunities'), pr_revision=None, pr_num=None)

## Meetings

In [18]:
MEETINGS_DIR = Path(".").resolve().parent / "data_cache" / "meetings"

APPLICATION_KEYS = [
    "application_id", "award_id", "title", "lead_applicant", "lead_organisation",
    "outcome", "score", "rank", "rank_group", "awarded_amount", "route",
    "opportunity", "panel", "notes",
]
PANELLIST_KEYS = ["name", "organisation", "role"]

# Collect all meeting JSON files, normalising keys across councils
records = []
for council_dir in sorted(MEETINGS_DIR.iterdir()):
    if not council_dir.is_dir():
        continue
    for f in sorted(council_dir.glob("*.json")):
        with open(f) as fh:
            meeting = json.load(fh)
        records.append({
            "council": meeting.get("council"),
            "meeting_name": meeting.get("meeting_name"),
            "meeting_reference": str(meeting["meeting_reference"]) if meeting.get("meeting_reference") is not None else None,
            "meeting_convenor": meeting.get("meeting_convenor"),
            "year": meeting.get("year"),
            "meeting_start": meeting.get("meeting_start"),
            "meeting_end": meeting.get("meeting_end"),
            "notes": meeting.get("notes"),
            "source": meeting.get("source"),
            "applications": [
                {k: a.get(k) for k in APPLICATION_KEYS}
                for a in meeting.get("applications", [])
            ],
            "panellists": [
                {k: p.get(k) for k in PANELLIST_KEYS}
                for p in meeting.get("panellists", [])
            ],
        })

meetings_df = pl.DataFrame(records, infer_schema_length=len(records))
meetings_df

council,meeting_name,meeting_reference,meeting_convenor,year,meeting_start,meeting_end,notes,source,applications,panellists
str,str,str,str,i64,str,str,str,str,list[struct[14]],list[struct[3]]
"""AHRC""","""Care For the Future Large Gran…",null,null,2014,"""2014-06""","""2014-06""","""Note 1. This list does not inc…","""Care For the Future Large Gran…","[{""AH/M004430/1"",null,""The Antislavery Usable Past"",""Kevin Bales"",""University of Hull"",""Successful"",""5"",null,null,null,null,null,null,null}, {""AH/M004376/1"",null,""Assembling Alternative Futures for Heritage"",""Rodney Harrison"",""University College London"",""Successful"",""5"",null,null,null,null,null,null,null}, … {""AH/M004422/1"",null,null,null,null,""Unsuccessful"",""4"",null,null,null,null,null,null,null}]",[]
"""AHRC""","""Design Research Fellowships, J…",null,null,2014,"""2014-06""","""2014-06""","""Note 1. This list does not inc…","""Design Research Fellowships Ju…","[{""AH/M005348/1"",null,""Designing Innovative Interventions with People Living with Dementia"",""Paul Rodgers"",""Northumbria University"",""Successful"",""5"",null,null,null,null,null,null,null}, {""AH/M005445/1"",null,""Co-designing an evaluation framework for design in the context of policy"",""Lucy Kimbell"",""University of Brighton"",""Successful"",""4"",null,null,null,null,null,null,null}, … {""AH/M005305/1"",null,null,null,null,""Unsuccessful"",""3"",null,null,null,null,null,null,null}]",[]
"""AHRC""","""Fellowships Panel - June 2014""",null,null,2014,"""2014-06""","""2014-06""","""Note 1. This list does not inc…","""Fellowships Panel June 2014.xl…","[{""AH/L014912/1"",null,""Corruption in Britain c.1550-1850"",""Mark Knights"",""University of Warwick"",""Successful"",""6"",null,null,null,""Standard"",null,null,null}, {""AH/L014815/1"",null,""Science in the public sphere: Understanding the meanings of ""applied science"" in the era of war, industrial research and modernism, 1900-1939"",""Robert Bud"",""Science Museum Group"",""Successful"",""6"",null,null,null,""Standard"",null,null,null}, … {""AH/L015307/1"",null,null,null,null,""Unsuccessful"",""2"",null,null,null,null,null,null,null}]",[]
"""AHRC""","""Care For the Future Early Care…",null,null,2014,"""2014-07""","""2014-07""","""Note 1. This list does not inc…","""Care For the Future Early Care…","[{""AH/M006263/1"",null,""Troubled Waters, Stormy Futures: heritage in times of accelerated climate change"",""Sara Penrhyn Jones"",""Aberystwyth University"",""Successful"",""6"",null,null,null,null,null,null,null}, {""AH/M006174/1"",null,""The Family Archive: Exploring Family Identities, Memories and Stories Through Curated Personal Possessions"",""Vicky Crewe"",""Cardiff University"",""Successful"",""6"",null,null,null,null,null,null,null}, … {""AH/M006212/1"",null,null,null,null,""Unsuccessful"",""3"",null,null,null,null,null,null,null}]",[]
"""AHRC""","""Connected Communities: Address…",null,null,2014,"""2014-07""","""2014-07""","""Note 1. This list does not inc…","""Connected Communities Addressi…","[{""AH/M00595X/1"",null,""Where we are not: the contribution of disconnection, division and exclusion to imaginative (im)mobility"",""Vanessa Burholt"",""Swansea University"",""Successful"",""4"",null,null,null,null,null,null,null}, {""AH/M006050/1"",null,""Alternative Futures: Disability and Community"",""Martin Levinson"",""University of Exeter"",""Successful"",""4"",null,null,null,null,null,null,null}, … {""AH/M005984/1"",null,null,null,null,""Unsuccessful"",""2"",null,null,null,null,null,null,null}]",[]
…,…,…,…,…,…,…,…,…,…,…
"""STFC""","""Ernest Rutherford Fellowships …",null,null,2024,"""2024""","""2024""",null,"""Ernest-Rutherford-fellowships-…","[{""APP53104"",null,null,null,null,""Successful"",null,""1"",null,null,null,""Ernest Rutherford Fellowships"",null,""Declined Offer""}, {""APP50692"",null,null,null,null,""Successful"",null,""2"",null,null,null,""Ernest Rutherford Fellowships"",null,null}, … {""APP55062"",null,null,null

In [19]:
MEETINGS_DATA_CARD = """\
---
license: cc-by-nc-sa-4.0
language:
- en
tags:
- funding
- ukri
- research-funding
- panel-meetings
- peer-review
- uk-research
pretty_name: UKRI Panel Meetings and Attendance
size_categories:
- 1K<n<10K
---

# UKRI Panel Meetings and Attendance

A structured dataset of 1,450 competitive funding panel meetings across seven UKRI research councils (AHRC, BBSRC, EPSRC, ESRC, MRC, NERC, STFC), comprising 38,862 application outcomes and 6,951 panel attendance records.

This dataset accompanies the paper *"Demystifying Funding: Reconstructing a Unified Dataset of the UK Funding Lifecycle"* (NSLP 2026) and the [GtR Database repository](https://github.com/wrmthorne/GtR-Extended). The source data was collected as part of the [UKRI Panel Meetings and Attendance](https://github.com/wrmthorne/UKRI-Panel-Meetings-and-Attendance) project.

## Background

UKRI research councils are required to publish the outcomes of competitive funding panel meetings, but do so inconsistently across councils and over time. Publication formats range from spreadsheets and PDFs to Tableau dashboards with data exports disabled. This dataset consolidates these disparate sources into a single structured format, enabling analysis of funding decisions at the point where panels determine which proposals receive funding.

## Collection

Meeting outcomes and panel attendance were collected from a combination of the UKRI website, the Government Web Archive, and (where no alternative existed) Tableau dashboards. Collection was performed between **18 March 2026 and 22 March 2026**. Source formats vary by council:

- **ESRC**: Continuously updated spreadsheet on UKRI website
- **AHRC** (pre-2018): National Archives; (post-2018): Tableau dashboard
- **BBSRC, MRC, NERC, STFC**: Combination of UKRI website, Government Web Archive, PDFs, and spreadsheets
- **EPSRC**: Tableau dashboard
- **MRC** (2025 onward): Tableau dashboard

Panel attendance is recorded separately from application outcomes in most cases, and is not available for all meetings. Of the 1,450 meetings, 588 have successfully reconciled panel attendance records.

## Structure

Each row represents a single panel meeting. Applications and panellists are stored as lists of objects within each row.

### Meeting-Level Fields

| Field | Type | Description |
|---|---|---|
| `council` | string | UKRI research council code (e.g. AHRC, EPSRC) |
| `meeting_name` | string | Name of the panel meeting |
| `meeting_reference` | string | Panel reference identifier, where available |
| `meeting_convenor` | string | Name of the meeting convenor, where available |
| `year` | int | Year of the meeting |
| `meeting_start` | string | Start date of the meeting (ISO 8601 or year-month) |
| `meeting_end` | string | End date of the meeting (ISO 8601 or year-month) |
| `notes` | string | Notes from the source document (e.g. sift exclusions) |
| `source` | string | Source file or URL the data was extracted from |

### Application Fields (nested list: `applications`)

| Field | Type | Description |
|---|---|---|
| `application_id` | string | Application identifier |
| `award_id` | string | Award/grant identifier, where available |
| `title` | string | Application title, where available |
| `lead_applicant` | string | Name of the lead applicant, where available |
| `lead_organisation` | string | Lead applicant's organisation, where available |
| `outcome` | string | Funding decision (e.g. Funded, Unsuccessful) |
| `score` | string | Panel score, where available |
| `rank` | string | Rank within the panel, where available |
| `rank_group` | string | Rank group, where available |
| `awarded_amount` | string | Awarded funding amount, where available |
| `route` | string | Funding route/scheme, where available |
| `opportunity` | string | Opportunity name referenced by the application, where available |
| `panel` | string | Sub-panel name, where available |
| `notes` | string | Application-level notes, where available |

### Panellist Fields (nested list: `panellists`)

| Field | Type | Description |
|---|---|---|
| `name` | string | Panellist name |
| `organisation` | string | Panellist's affiliated organisation |
| `role` | string | Role on the panel (e.g. Member, Chair) |

## Limitations

- **Incomplete attendance**: Only 588 of 1,450 meetings have reconciled panel attendance, due to attendance being recorded separately or not at all by many councils.
- **Inconsistent fields**: Not all fields are populated for all councils. Available metadata varies significantly by council and time period (e.g. scores and ranks are not always provided).
- **Tableau restrictions**: EPSRC, post-2018 AHRC, and MRC from 2025 onward publish exclusively through Tableau with data exports disabled, limiting what could be extracted.
- **AHRC post-2018**: Unavailable at the time of writing following a report to UKRI concerning unintentionally published applicant names, affiliations, and project titles for unsuccessful proposals.
- **Date granularity**: Some meetings only have year-month precision for start/end dates.
"""

In [20]:
meetings_dataset = Dataset.from_polars(meetings_df)
meetings_dataset.push_to_hub(MEETINGS_REPO_ID, token=os.environ["HF_TOKEN_WRITE"])

card = DatasetCard(MEETINGS_DATA_CARD)
card.push_to_hub(MEETINGS_REPO_ID, token=os.environ["HF_TOKEN_WRITE"])

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/wrmthorne/ukri-panel-meetings/commit/42a47904f462ca6cb3b74b223e095f0fd4ae1cb7', commit_message='Upload README.md with huggingface_hub', commit_description='', oid='42a47904f462ca6cb3b74b223e095f0fd4ae1cb7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wrmthorne/ukri-panel-meetings', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wrmthorne/ukri-panel-meetings'), pr_revision=None, pr_num=None)